In [ ]:
from xact.log.config import log_manager
from xact.config.gen import config

log_manager.config(level=20)


from xact.llm import llm_client


print(config.XACT_LLM_MODEL)

from xact.llm.llm import LLM


# log.enable(True)
# config.XACT_LLM_MODEL = "qwen2.5:1.5b "

log = log_manager.init(__name__)

llm = LLM("llama3.2:3b")

Feb/09 14:37:26 |   xact.log.config     | INFO     | log initilized
Feb/09 14:37:26 |   xact.log.config     | XACT_STREAM | log streaming initilized
Feb/09 14:37:26 |   xact.config.gen     | INFO     | Config loaded from .env:.env json_path : config.json


qwen2.5:1.5b


In [2]:
llm.list()

['deepseek-r1:7b',
 'deepseek-r1:1.5b',
 'deepseek-coder:1.3b',
 'granite3.1-moe:latest',
 'llama3.2:3b',
 'codegemma:2b',
 'granite3.1-dense:2b',
 'gemma2:2b',
 'llava:latest',
 'llama3.2:1b',
 'nomic-embed-text:latest',
 'mistral:latest',
 'nemotron-mini:latest',
 'qwen2.5-coder:1.5b',
 'qwen2.5:3b',
 'qwen2.5:1.5b']

In [2]:
log_manager.loggers

{'xact.log.config': <Logger xact.log.config (INFO)>,
 'xact-stream': <Logger xact-stream (INFO)>,
 'xact': <Logger xact (INFO)>,
 'xact.config.gen': <Logger xact.config.gen (INFO)>,
 'xact.llm.llm': <Logger xact.llm.llm (DEBUG)>,
 '__main__': <Logger __main__ (DEBUG)>}

In [3]:
config.XACT_LLM_MODEL = "qwen2.5:1.5b "

In [ ]:
log_manager.ext_enable()
log_manager.enable()

Feb/09 14:46:10 |     xact.llm.llm      | INFO     | llm out generated


Hello! How can I assist you today?


In [ ]:
def get_data(year: int, month: int, day: int):
    """return data for given date"""
    return "data"


from xact.data.data import DataX


from xact.llm.tool import gen_function_schema, tool
from xact.utils.gen import gen_datetime, gen_uuid

from xact.flow.router.router import RouteData
from xact.flow.router.router import Router

tools = [
    get_data,
    gen_uuid,
    tool(func=gen_datetime),
]
routedata = RouteData()
routedata.embed(data_list=tools + [DataX(content="data")])
routedata.data_embd_str

[' get_data description : return data for given date function : [{"type": "function", "function": {"name": "get_data", "description": "return data for given date", "parameters": {"type": "object", "properties": {"year": {"type": "integer"}, "month": {"type": "integer"}, "day": {"type": "integer"}}, "required": ["year", "month", "day"]}}}]',
 ' gen_uuid description : generate uuid function : [{"type": "function", "function": {"name": "gen_uuid", "description": "generate uuid", "parameters": {"type": "object", "properties": {}, "required": []}}}]',
 'gen_datetime description : give local_utc or utc time function : [{"type": "function", "function": {"name": "gen_datetime", "description": "give local_utc or utc time", "parameters": {"type": "object", "properties": {"local_utc": {"type": "boolean"}}, "required": []}}}]',
 'data']

In [ ]:
res = Router.route_vector(prompt="give some unique id", route_data=routedata)
res

{'idx': [1, 3, 2, 0],
 'score': [0.4574908990475241,
  0.452059762787031,
  0.4243961503100083,
  0.41752060200895785],
 'data': [<function xact.utils.gen.gen_uuid()>,
  DataX(uid=UUID('bf83694a-7852-4fb0-ae76-9e182cd13c81'), cid=None, flow_mode='prompt', role='xact', content='data', content_type='str', md_content=None, metadata=None, tags=None, description=None, source=None, embed=None, embed_id=None, created_at=datetime.datetime(2025, 2, 9, 0, 39, 7, 926081), time_triggers=None),
  <function __main__.get_data(year: int, month: int, day: int)>],
 'data_embed': []}

In [ ]:
llm.model.var = "llama3.2:1b"
messages = [
    {
        "role": "system",
        "content": f"you are ox-ai helpful ai assistant you are excelent at resonaing and assisting in any tasks you are given with list of tools u need to pick whihc tool is best suited for the task toos : {routedata.data_embd_str}   give the tool name as output in json format",
    },
    {
        "role": "user",
        "content": "task : to find an unique id \n\n give the tool name that can performa the task tool_name : 'name of the tool' ",
    },
]
from pydantic import BaseModel


class ToolResponse(BaseModel):
    toolname: str


res = llm.generate(messages=messages, response_format=ToolResponse)

print(res)

Feb/09 15:47:17 |     xact.llm.llm      | INFO     | llm out generated


toolname='gen_uuid'


In [ ]:
def my_function(param1: str, param2: int, param3: bool = True) -> str:
    """
    This function does something.

    Args:
        param1: A string parameter.  (Description of param1)
        param2: An integer parameter. (Description of param2)
        param3: An optional boolean parameter. Defaults to True. (Description of param3)

    Returns:
        A string.
    """
    # ... function code ...
    return "result"

In [ ]:
mt = tool(func=my_function)
mt.get_schema()

{'type': 'function',
 'function': {'name': 'my_function',
  'description': 'This function does something.\n\n    Args:\n        param1: A string parameter.  (Description of param1)\n        param2: An integer parameter. (Description of param2)\n        param3: An optional boolean parameter. Defaults to True. (Description of param3)\n\n    Returns:\n        A string.',
  'parameters': {'type': 'object',
   'properties': {'param1': {'type': 'string'},
    'param2': {'type': 'integer'},
    'param3': {'type': 'boolean'}},
   'required': ['param1', 'param2']}}}

In [1]:
import json
from typing import Dict, List
from xact.llm.tool import tool, Tool
from xact.utils.gen import gen_uuid
from xact.log.config import log_manager
from xact.flow.router.router import RouteData,Router
from xact.llm.llm import LLM

llm = LLM()

log = log_manager.init(__name__)


class Flow:
    def __init__(
        self,
        mode: str,
    ):
        self.mode = mode


class Action:
    _instances = {}
    _route_tools = RouteData()


    def __new__(cls, name=None, *args, **kwargs):
        if name is None and "func" in kwargs and callable(kwargs["func"]):
            name = kwargs[
                "func"
            ].__name__  # Auto-assign function name if `name` is not provided

        
        if name in cls._instances:
            return cls._instances[name]  # Return existing instance

        if name:
            instance = super().__new__(cls)
            cls._instances[name] = instance  # Register instance
            return instance
        else :
            raise "no name or function proviede"
        

    def __init__(
        self, name: str = None, description: str = None, func: callable = None
    ):
        if not hasattr(self, "name"):  # Ensure __init__ only runs once per instance
            if func:
                self.func = func
                self.tool = tool(description=description, func=func)
                self.name = name or self.tool.fun_schema["function"]["name"]
                self.description = self.tool.fun_schema["function"]["description"]
                self.uid = gen_uuid()
                self._instances[self.name] = self
                self._route_tools.embed(data_list=[self.tool])

    @classmethod
    def get_actions(cls):
        return cls._instances  # Return all registered instances

    def act(
        self,
        prompt: str = None,
        messages: List[Dict] = None,
        model: str = None,
        loop:bool=False,
        kwargs: dict = None,
     
    ):
        log.info(f"execution of {self.name} Action")
        act_res = self.nact(prompt=prompt,messages=messages,model=model,tools=[self.tool],top_n=1,kwargs=kwargs,loop=loop,)
        return act_res
    

    @staticmethod
    def nact(
        prompt: str = None,
        messages: List[Dict] = None,
        model: str = None,
        func_list:List[callable]=None,
        tools: List[Tool] = None,
        route_tools:RouteData=None,
        top_n:int=5,
        loop:bool = False,
        kwargs: dict = None,
        
    ):
        messages = messages or [
            {"role": "user", "content": prompt.strip()},
        ]
        prompt = prompt or messages[-1]["content"]

        if func_list:
            tools
            for func in func_list:
                tools.append(tool(func=func))
        if tools:
            route_tools =RouteData()
            route_tools.embed(data_list=tools)

        if not route_tools:
            if Action._route_tools.data_list:
                route_tools = Action._route_tools
            else:
                return "no actions in registery"

        tools_schema = []
        input_tools=[]

        if not len(route_tools.data_list) ==1:


            route_res = Router.route_vector(prompt=prompt,route_data=route_tools)

            input_tools = route_res["data"][0:top_n]
            input_tools.reverse()
  
        
        else:
            if not route_tools.data_list[0].fun_schema["function"]["parameters"]["properties"]:
                try :
                    fun_res = route_tools.data_list[0].run()
                    return fun_res
                except Exception as e :
                    return str(e)
            else:
                input_tools = route_tools.data_list
            
        for tool_obj in input_tools:
            tools_schema.append(tool_obj.get_schema())

        llm_out = llm.generate(model=model, messages=messages, tools=tools_schema)

        tool_calls= llm_out["completion"].choices[0].message.tool_calls
        if not isinstance(tool_calls,list): 
            content = llm_out["completion"].choices[0].message.content
            return content if content else "action is not executed"
  
        for tool_obj in input_tools:
            if tool_obj.fun_schema["function"]["name"] == tool_calls[0].function.name:
                try :
                    arguments = json.loads(tool_calls[0].function.arguments)
                    fun_res = tool_obj.run(**arguments)
                    return fun_res
                except Exception as e :
                    return str(e)
            
            



Feb/10 00:17:16 |   xact.log.config     | INFO     | log initilized
Feb/10 00:17:16 |   xact.log.config     | XACT_STREAM | log streaming initilized
Feb/10 00:17:17 |   xact.config.gen     | INFO     | Config loaded from .env:.env json_path : config.json


In [10]:
# Example Usage:
# 
def shutdown():
    return "shutting down"


shutdown_action = Action(
    func=shutdown
)  # No name provided, should auto-assign "shutdown"

print(shutdown_action.name)  # Output: "shutdown"


def open_apps(name: str):
    return f"{name} app opened"


open_apps_action = Action(
    name="open_apps", description="open apps from the given name", func=open_apps
)

print(open_apps_action.func(name="chrome"))  # Output: "chrome app opened"
print(Action.get_actions())

shutdown
chrome app opened
{'shutdown': <__main__.Action object at 0x7ff6c4b612d0>, 'open_apps': <__main__.Action object at 0x7ff6c4b623d0>}


In [ ]:
shutdown_action.act(prompt="shutdown the system")

Feb/10 00:17:39 |       __main__        | INFO     | execution of shutdown Action
Feb/10 00:17:39 |    xact.llm.embed     | INFO     | llm embed generated
Feb/10 00:17:39 |    xact.llm.tool      | INFO     | executiong tool : shutdown


'shutting down'

In [6]:
Action.nact(prompt="open youtube")

Feb/10 00:17:45 |    xact.llm.embed     | INFO     | llm embed generated
Feb/10 00:18:17 |     xact.llm.llm      | INFO     | llm out generated
Feb/10 00:18:17 |    xact.llm.tool      | INFO     | executiong tool : open_apps


'YouTube app opened'

In [7]:
Action.nact(prompt="shutdown the system")

Feb/10 00:18:24 |    xact.llm.embed     | INFO     | llm embed generated
Feb/10 00:18:28 |     xact.llm.llm      | INFO     | llm out generated
Feb/10 00:18:28 |    xact.llm.tool      | INFO     | executiong tool : shutdown


'shutting down'

In [ ]:
from xact.llm.tool import tool


@tool(
    description="mf",
    param_description={
        "param1": "A string parameter.",
        "param2": "An integer parameter.",
        "param3": "An optional boolean parameter. Defaults to True.",
    },
)
def my_f(
    hhh: str, param2: int, param3: bool = True
) -> str:  # Type hints for parameters and return value
    """
    This function does something.

    Args:
        param1: A string parameter.  (Description of param1)
        param2: An integer parameter. (Description of param2)
        param3: An optional boolean parameter. Defaults to True. (Description of param3)

    Returns:
        A string.
    """
    # ... function code ...
    return "result"


def my_ff(
    hhh: str, param2: int, param3: bool = True
) -> str:  # Type hints for parameters and return value
    """
    This function does something.

    Args:
        param1: A string parameter.  (Description of param1)
        param2: An integer parameter. (Description of param2)
        param3: An optional boolean parameter. Defaults to True. (Description of param3)

    Returns:
        A string.
    """
    # ... function code ...
    return "result"

In [ ]:
print(my_f(1, 2))

Feb/09 00:39:25 |    xact.llm.tool      | INFO     | executiong tool : my_f


result


In [ ]:
print(tool(func=my_f)(1, 2))

Feb/09 00:39:25 |    xact.llm.tool      | INFO     | executiong tool : my_f


result


In [ ]:
from datetime import datetime


tools = [
    gen_function_schema(get_data),
    gen_function_schema(gen_uuid),
    gen_function_schema(gen_datetime),
]

model = "qwen2.5:1.5b"
model = "llama3.2:3b"
# model = "granite3.1-dense:2b"
completion = llm_client.chat.completions.create(
    model=model,
    messages=[
        {"role": "user", "content": "What the best party [mumbai bangaore chennai]"}
    ],
    tools=tools,
    tool_choice="required",
    temperature=0,
)

print(completion.choices[0].message.tool_calls)
for cmp in completion.choices[0]:
    print(cmp)  # .choices[0].message.tool_calls)
    print("2222")

[ChatCompletionMessageToolCall(id='call_fqshgr06', function=Function(arguments='{"day":"31","month":"12","year":"2022"}', name='get_data'), type='function', index=0)]
('finish_reason', 'tool_calls')
2222
('index', 0)
2222
('logprobs', None)
2222
('message', ChatCompletionMessage(content='', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_fqshgr06', function=Function(arguments='{"day":"31","month":"12","year":"2022"}', name='get_data'), type='function', index=0)]))
2222


In [15]:
from datetime import date, timedelta

# Get the current date
today = date.today()
print("Today:", today)

# Add one day
tomorrow = today + timedelta(days=1)
print("Tomorrow:", tomorrow)

# Add multiple days
in_three_days = today + timedelta(days=3)
print("In three days:", in_three_days)

# Subtract days (go to the past)
yesterday = today - timedelta(days=1)
print("Yesterday:", yesterday)

# Working with specific dates:
specific_date = date(2024, 1, 1)  # January 1, 2024
next_day = specific_date + timedelta(days=1)
print(f"The day after {specific_date}: {next_day}")

# Example with months/years:
# Note: timedelta only works with days, seconds, microseconds, milliseconds, minutes, hours, and weeks.
# For month or year arithmetic, you need a different approach (see below).

Today: 2025-02-09
Tomorrow: 2025-02-10
In three days: 2025-02-12
Yesterday: 2025-02-08
The day after 2024-01-01: 2024-01-02


In [ ]:
from xact.config.gen import Config

In [17]:
print(res)

{'idx': [1, 3, 2, 0], 'score': [0.4574908990475241, 0.452059762787031, 0.4243961503100083, 0.41752060200895785], 'data': [<function gen_uuid at 0x7fd7ff717060>, DataX(uid=UUID('bf83694a-7852-4fb0-ae76-9e182cd13c81'), cid=None, flow_mode='prompt', role='xact', content='data', content_type='str', md_content=None, metadata=None, tags=None, description=None, source=None, embed=None, embed_id=None, created_at=datetime.datetime(2025, 2, 9, 0, 39, 7, 926081), time_triggers=None), <xact.llm.tool.Tool object at 0x7fd7ff6eb790>, <function get_data at 0x7fd7feb8c5e0>], 'data_embed': []}


In [18]:
config.AUDIO_FORMAT

8